# Bài 3 — Học máy với dữ liệu ảnh (Oxford-IIIT Pet)

**Môn học:** Học máy (Machine Learning) — Học kỳ I, năm học 2026-2027  
**Khoa Khoa học và Kỹ thuật Máy tính, Trường Đại học Bách Khoa, ĐHQG-HCM**  
**GVHD:** *[điền tên giảng viên]*  
**Nhóm:** *[tên nhóm]* — *[Họ tên – MSSV – email]* (2-3 thành viên)  
**Mã nguồn GitHub:** *[điền link repo nếu có]*

> **Hướng dẫn chạy:** chọn *Runtime → Change runtime type → T4 GPU* (khuyến nghị), sau đó *Runtime → Run all*.
> Notebook tự cài thư viện, tự tải dữ liệu công khai về máy chủ Colab, **không** mount Google Drive.

## 0. Cài đặt môi trường & nạp module

Các module Python hỗ trợ (`modules/*.py`) được ghi trực tiếp ra máy chủ Colab bằng `%%writefile` (cùng nội dung với thư mục `modules/` trong bài nộp / GitHub), nên notebook chạy độc lập mà không cần Drive.

In [ ]:
!pip -q install scikit-image scikit-learn seaborn h5py
!mkdir -p modules reports/figures features data

In [ ]:
%%writefile modules/common.py
"""Tiện ích dùng chung cho cả 3 bài: seed, đánh giá, vẽ hình, lưu kết quả/đặc trưng."""
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

REPORT_DIR = "reports"
FIG_DIR = os.path.join(REPORT_DIR, "figures")
FEATURE_DIR = "features"


def ensure_dirs():
    for d in (REPORT_DIR, FIG_DIR, FEATURE_DIR, "data"):
        os.makedirs(d, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def evaluate(y_true, y_pred, average: str = "macro") -> dict:
    """Accuracy, precision, recall, F1 (average = 'macro' | 'weighted' | 'binary')."""
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=average, zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": p,
        "recall": r,
        "f1": f1,
    }


def save_fig(name: str, show: bool = True):
    """Lưu hình vào reports/figures/<name>.png (dùng cho báo cáo) và hiển thị."""
    ensure_dirs()
    path = os.path.join(FIG_DIR, f"{name}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()
    return path


def plot_confusion(y_true, y_pred, labels=None, title="Confusion matrix", name=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    n = cm.shape[0]
    size = max(5, min(14, n * 0.45))
    fig, ax = plt.subplots(figsize=(size, size * 0.85))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Dự đoán")
    ax.set_ylabel("Thực tế")
    if labels is not None and n <= 25:
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046)
    if name:
        save_fig(name)
    else:
        plt.show()


class ResultLogger:
    """Gom kết quả thí nghiệm thành bảng và lưu CSV để báo cáo dùng lại."""

    def __init__(self, name: str):
        self.name = name
        self.rows = []

    def add(self, **kwargs):
        self.rows.append(kwargs)

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows)

    def save(self):
        ensure_dirs()
        df = self.frame()
        df.to_csv(os.path.join(REPORT_DIR, f"results_{self.name}.csv"), index=False)
        return df


class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *a):
        self.elapsed = time.time() - self.t0


def save_features(name: str, array: np.ndarray, fmt: str = "npy"):
    """Lưu đặc trưng vào features/<name>.npy (hoặc .h5)."""
    ensure_dirs()
    path = os.path.join(FEATURE_DIR, f"{name}.{fmt}")
    if fmt == "npy":
        np.save(path, array)
    elif fmt == "h5":
        import h5py

        with h5py.File(path, "w") as f:
            f.create_dataset("features", data=array, compression="gzip")
    else:
        raise ValueError(fmt)
    return path


def load_features(name: str, fmt: str = "npy"):
    path = os.path.join(FEATURE_DIR, f"{name}.{fmt}")
    if fmt == "npy":
        return np.load(path)
    import h5py

    with h5py.File(path, "r") as f:
        return f["features"][:]


def save_json(name: str, obj: dict):
    ensure_dirs()
    with open(os.path.join(REPORT_DIR, f"{name}.json"), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=float)


def download(url: str, dest: str):
    """Tải file công khai về dest (bỏ qua nếu đã có)."""
    import urllib.request

    if os.path.exists(dest):
        return dest
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    print(f"Đang tải {url} ...")
    urllib.request.urlretrieve(url, dest)
    return dest

In [ ]:
%%writefile modules/image.py
"""Bài 3 - Dữ liệu ảnh: Oxford-IIIT Pet (37 giống chó/mèo, ~7.4k ảnh).

Nguồn công khai: https://www.robots.ox.ac.uk/~vgg/data/pets/ (tải qua torchvision.datasets.OxfordIIITPet).
Đặc trưng truyền thống: HOG (+ PCA).
Đặc trưng hiện đại: ResNet50 / EfficientNet-B0 / ViT-B/16 pretrained (ImageNet) -> lưu .npy.
"""
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

from common import Timer, evaluate, save_features, save_fig

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


# ----------------------------------------------------------------------------
# Dữ liệu
# ----------------------------------------------------------------------------
def load_pets(root: str = "data"):
    """Trả về (train_ds, test_ds, class_names). Tập train = 'trainval' (3680), test = 'test' (3669)."""
    from torchvision.datasets import OxfordIIITPet

    tr = OxfordIIITPet(root, split="trainval", target_types="category", download=True)
    te = OxfordIIITPet(root, split="test", target_types="category", download=True)
    return tr, te, list(tr.classes)


def labels_of(ds):
    return np.array(ds._labels)


# ----------------------------------------------------------------------------
# EDA
# ----------------------------------------------------------------------------
def run_eda(train_ds, test_ds, class_names, sample_n: int = 2000, seed: int = 42):
    y_tr, y_te = labels_of(train_ds), labels_of(test_ds)
    print(f"Train: {len(train_ds)} ảnh | Test: {len(test_ds)} ảnh | số lớp: {len(class_names)}")

    counts = pd.Series(y_tr).value_counts().sort_index()
    plt.figure(figsize=(12, 4))
    plt.bar(range(len(counts)), counts.values)
    plt.xticks(range(len(counts)), [class_names[i] for i in counts.index], rotation=90, fontsize=7)
    plt.title("Số ảnh theo lớp (train)")
    save_fig("img_class_dist")
    print(f"Số ảnh / lớp (train): min={counts.min()}, max={counts.max()}, mean={counts.mean():.1f}")

    # Kích thước, kênh màu, tỉ lệ khung hình (lấy mẫu ngẫu nhiên để nhanh)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(train_ds), min(sample_n, len(train_ds)), replace=False)
    sizes, modes = [], Counter()
    for i in idx:
        with Image.open(train_ds._images[i]) as im:
            sizes.append(im.size)
            modes[im.mode] += 1
    w, h = np.array(sizes).T
    print(f"Chế độ màu (mẫu {len(idx)} ảnh): {dict(modes)}")
    print(f"Rộng: min={w.min()} max={w.max()} median={np.median(w):.0f} | "
          f"Cao: min={h.min()} max={h.max()} median={np.median(h):.0f}")
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].scatter(w, h, s=4, alpha=0.4)
    axes[0].set_title("Rộng vs Cao (px)")
    axes[1].hist(w / h, bins=40)
    axes[1].set_title("Tỉ lệ khung hình (W/H)")
    axes[2].bar(list(modes.keys()), list(modes.values()))
    axes[2].set_title("Số kênh màu (mode PIL)")
    save_fig("img_size_dist")

    # Ảnh mẫu
    fig, axes = plt.subplots(3, 6, figsize=(14, 7))
    pick = rng.choice(len(train_ds), 18, replace=False)
    for ax, i in zip(axes.ravel(), pick):
        ax.imshow(Image.open(train_ds._images[i]).convert("RGB"))
        ax.set_title(class_names[y_tr[i]], fontsize=8)
        ax.axis("off")
    save_fig("img_samples")

    # Cường độ điểm ảnh trung bình theo kênh (ảnh đã resize nhỏ)
    chans = []
    for i in idx[:500]:
        a = np.asarray(Image.open(train_ds._images[i]).convert("RGB").resize((64, 64)), dtype=np.float32) / 255
        chans.append(a.reshape(-1, 3).mean(0))
    chans = np.array(chans)
    print("Trung bình RGB:", chans.mean(0).round(3), "| std:", chans.std(0).round(3))
    plt.figure(figsize=(5, 4))
    for c, col in enumerate(("r", "g", "b")):
        plt.hist(chans[:, c], bins=30, alpha=0.5, color=col, label=col.upper())
    plt.legend()
    plt.title("Phân phối cường độ trung bình theo kênh")
    save_fig("img_channel_dist")


# ----------------------------------------------------------------------------
# Đặc trưng truyền thống: HOG
# ----------------------------------------------------------------------------
def _hog_one(path, size, ppc, cpb, orientations):
    from skimage.feature import hog

    im = Image.open(path).convert("L").resize((size, size))
    return hog(np.asarray(im), orientations=orientations, pixels_per_cell=(ppc, ppc),
               cells_per_block=(cpb, cpb), block_norm="L2-Hys")


def hog_features(ds, size=128, pixels_per_cell=16, cells_per_block=2, orientations=9, n_jobs=-1):
    from joblib import Parallel, delayed

    feats = Parallel(n_jobs=n_jobs)(
        delayed(_hog_one)(p, size, pixels_per_cell, cells_per_block, orientations) for p in ds._images)
    return np.asarray(feats, dtype=np.float32)


# ----------------------------------------------------------------------------
# Đặc trưng hiện đại: pretrained backbone
# ----------------------------------------------------------------------------
def get_backbone(name: str):
    """Trả về (model đã bỏ lớp phân loại, kích thước đầu vào mặc định, dim đặc trưng)."""
    import torch.nn as nn
    import torchvision.models as tvm

    if name == "resnet50":
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Identity()
        return m, 224
    if name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier = nn.Identity()
        return m, 224
    if name == "vit_b_16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.IMAGENET1K_V1)
        m.heads = nn.Identity()
        return m, 224  # ViT-B/16 yêu cầu đúng 224x224
    if name == "vgg16":
        m = tvm.vgg16(weights=tvm.VGG16_Weights.IMAGENET1K_V1)
        m.classifier = nn.Sequential(*list(m.classifier.children())[:-1])  # 4096-d
        return m, 224
    raise ValueError(name)


def extract_deep_features(ds, name: str, img_size: int = None, batch_size: int = 64, num_workers: int = 2):
    """Resize ảnh -> chuẩn hoá ImageNet -> forward qua backbone pretrained -> vector đặc trưng."""
    import torch
    import torchvision.transforms as T
    from torch.utils.data import DataLoader, Dataset

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, default_size = get_backbone(name)
    size = img_size or default_size
    if name == "vit_b_16" and size != 224:
        raise ValueError("ViT-B/16 chỉ hỗ trợ 224x224")
    model = model.to(dev).eval()

    tf = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

    class _DS(Dataset):
        def __len__(self):
            return len(ds._images)

        def __getitem__(self, i):
            return tf(Image.open(ds._images[i]).convert("RGB"))

    dl = DataLoader(_DS(), batch_size=batch_size, num_workers=num_workers)
    out = []
    with torch.no_grad():
        for xb in dl:
            xb = xb.to(dev)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                f = model(xb)
            out.append(f.float().cpu().numpy())
    return np.concatenate(out)


def extract_and_save(ds_train, ds_test, name: str, img_size: int = None):
    with Timer() as t:
        tr = extract_deep_features(ds_train, name, img_size)
        te = extract_deep_features(ds_test, name, img_size)
    tag = f"{name}_{img_size or 224}"
    save_features(f"img_{tag}_train", tr)
    save_features(f"img_{tag}_test", te)
    print(f"{tag}: train {tr.shape}, test {te.shape} ({t.elapsed:.0f}s) -> features/img_{tag}_*.npy")
    return tr, te


# ----------------------------------------------------------------------------
# Bộ phân loại
# ----------------------------------------------------------------------------
def make_classifier(name: str, **kw):
    if name == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=kw.get("C", 1.0)))
    if name == "linsvm":
        return make_pipeline(StandardScaler(), LinearSVC(C=kw.get("C", 0.01), max_iter=20000))
    if name == "svm_rbf":
        return make_pipeline(StandardScaler(), SVC(kernel="rbf", C=kw.get("C", 10.0), gamma="scale"))
    if name == "rf":
        return RandomForestClassifier(n_estimators=kw.get("n_estimators", 300), n_jobs=-1, random_state=42)
    raise ValueError(name)


def fit_eval(clf, Xtr, ytr, Xte, yte, average="macro"):
    with Timer() as t:
        clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    res = evaluate(yte, pred, average=average)
    res["fit_seconds"] = round(t.elapsed, 2)
    return res, pred


def pca_reduce(Xtr, Xte, n_components=0.95, seed=42):
    pca = PCA(n_components=n_components, random_state=seed).fit(Xtr)
    return pca.transform(Xtr), pca.transform(Xte), pca


# ----------------------------------------------------------------------------
# Fine-tuning đầu-cuối (điểm cộng pipeline học sâu)
# ----------------------------------------------------------------------------
def finetune_cnn(train_ds, test_ds, n_classes, arch="resnet50", epochs=5, batch_size=32,
                 lr=3e-4, img_size=224, num_workers=2, seed=42):
    import torch
    import torch.nn as nn
    import torchvision.models as tvm
    import torchvision.transforms as T
    from torch.utils.data import DataLoader, Dataset

    torch.manual_seed(seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if arch == "resnet50":
        model = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, n_classes)
    elif arch == "resnet18":
        model = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, n_classes)
    else:
        raise ValueError(arch)
    model = model.to(dev)

    norm = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    tf_train = T.Compose([T.RandomResizedCrop(img_size, scale=(0.7, 1.0)), T.RandomHorizontalFlip(),
                          T.ToTensor(), norm])
    tf_test = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), norm])

    class _DS(Dataset):
        def __init__(self, base, tf):
            self.base, self.tf = base, tf

        def __len__(self):
            return len(self.base._images)

        def __getitem__(self, i):
            return self.tf(Image.open(self.base._images[i]).convert("RGB")), self.base._labels[i]

    tr_dl = DataLoader(_DS(train_ds, tf_train), batch_size=batch_size, shuffle=True,
                       num_workers=num_workers, drop_last=True)
    te_dl = DataLoader(_DS(test_ds, tf_test), batch_size=batch_size * 2, num_workers=num_workers)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(tr_dl))
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    with Timer() as t:
        for ep in range(epochs):
            model.train()
            tot = 0.0
            for xb, yb in tr_dl:
                xb, yb = xb.to(dev), yb.to(dev)
                opt.zero_grad()
                with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                    loss = loss_fn(model(xb), yb)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
                tot += loss.item()
            print(f"  epoch {ep + 1}/{epochs} loss={tot / len(tr_dl):.4f}")
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in te_dl:
            preds.append(model(xb.to(dev)).argmax(1).cpu().numpy())
    pred = np.concatenate(preds)
    res = evaluate(labels_of(test_ds), pred, average="macro")
    res["fit_seconds"] = round(t.elapsed, 2)
    return res, pred

In [ ]:
%%writefile modules/dl.py
"""Phần học sâu (điểm cộng): MLP PyTorch cho vector đặc trưng + vòng huấn luyện dùng chung."""
import copy

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from common import Timer, evaluate


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128), dropout=0.3):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp(X_train, y_train, X_test, y_test, hidden=(256, 128), dropout=0.3, lr=1e-3,
              weight_decay=1e-4, epochs=60, batch_size=256, patience=8, average="macro",
              seed=42, verbose=True):
    """Huấn luyện MLP với early stopping trên 10% tập train. Trả về (metrics, model)."""
    torch.manual_seed(seed)
    dev = get_device()
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_train = np.asarray(y_train)
    n_classes = int(max(y_train.max(), np.asarray(y_test).max()) + 1)

    X_tr, X_va, y_tr, y_va = train_test_split(
        X_train, y_train, test_size=0.1, stratify=y_train, random_state=seed)
    Xt, yt = torch.from_numpy(X_tr), torch.from_numpy(y_tr).long()
    Xv, yv = torch.from_numpy(X_va).to(dev), torch.from_numpy(y_va).long().to(dev)

    model = MLP(X_tr.shape[1], n_classes, hidden, dropout).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    best, best_state, bad = float("inf"), None, 0

    with Timer() as t:
        for ep in range(epochs):
            model.train()
            perm = torch.randperm(len(Xt))
            for i in range(0, len(perm), batch_size):
                idx = perm[i:i + batch_size]
                if len(idx) < 2:
                    continue
                xb, yb = Xt[idx].to(dev), yt[idx].to(dev)
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                opt.step()
            model.eval()
            with torch.no_grad():
                vl = loss_fn(model(Xv), yv).item()
            if vl < best - 1e-4:
                best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
                if bad >= patience:
                    break
            if verbose and (ep + 1) % 10 == 0:
                print(f"  epoch {ep + 1:3d} val_loss={vl:.4f}")
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_test).to(dev))
        pred = logits.argmax(1).cpu().numpy()
    res = evaluate(y_test, pred, average=average)
    res["fit_seconds"] = round(t.elapsed, 2)
    res["epochs_run"] = ep + 1
    return res, model, pred

In [ ]:
import sys, warnings
sys.path.insert(0, 'modules')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_seed(42); ensure_dirs()
import image as IM
import dl
import torch
print('GPU:', torch.cuda.is_available())

## 1. Bài toán & dữ liệu

- **Bài toán:** phân loại ảnh thú cưng thành **37 giống** (25 giống chó, 12 giống mèo) — bài toán *fine-grained*.
- **Dữ liệu:** *Oxford-IIIT Pet* — https://www.robots.ox.ac.uk/~vgg/data/pets/ (tải qua `torchvision.datasets.OxfordIIITPet`).
  Tập train = `trainval` (3,680 ảnh), tập test = `test` (3,669 ảnh), ~200 ảnh/lớp, ảnh màu kích thước và tỉ lệ khác nhau.
- **Vì sao phù hợp:** đủ lớn và đa dạng, chạy được trên Colab (ảnh JPEG, ~800MB), có các giống rất giống nhau ⇒ phân tích lỗi thú vị.

> ⚠️ Đề yêu cầu các nhóm không trùng tập dữ liệu và cần thống nhất với giảng viên. Đổi dataset: sửa `load_pets()` trong `modules/image.py`.

In [ ]:
train_ds, test_ds, class_names = IM.load_pets('data')
y_train, y_test = IM.labels_of(train_ds), IM.labels_of(test_ds)
print(len(train_ds), len(test_ds), len(class_names))

## 2. EDA ảnh

In [ ]:
IM.run_eda(train_ds, test_ds, class_names)

**Nhận xét EDA** *(chỉnh theo kết quả thực tế)*:
- Số ảnh/lớp khá cân bằng (~100 ảnh/lớp ở train), không cần cân bằng lại.
- Ảnh đều là **RGB 3 kênh** (có thể có vài ảnh ở chế độ khác ⇒ luôn `convert('RGB')`).
- **Kích thước và tỉ lệ khung hình không đồng nhất** (chiều rộng ~200–4000 px) ⇒ bắt buộc **resize** về kích thước cố định trước khi trích đặc trưng.
- Các giống như *Staffordshire Bull Terrier* / *American Pit Bull Terrier* hay *Egyptian Mau* / *Bengal* rất giống nhau ⇒ dự kiến nhầm lẫn.

## 3. Pipeline truyền thống: HOG → (PCA) → bộ phân loại

HOG (Histogram of Oriented Gradients) trên ảnh xám resize `128×128`, `pixels_per_cell=16`, sau đó PCA giữ 95% phương sai.

In [ ]:
with Timer() as t:
    hog_tr = IM.hog_features(train_ds, size=128, pixels_per_cell=16)
    hog_te = IM.hog_features(test_ds, size=128, pixels_per_cell=16)
print(hog_tr.shape, hog_te.shape, f'{t.elapsed:.0f}s')
save_features('img_hog128_train', hog_tr); save_features('img_hog128_test', hog_te)
save_features('img_y_train', y_train); save_features('img_y_test', y_test)

In [ ]:
log = ResultLogger('image_traditional')
hog_pca_tr, hog_pca_te, pca = IM.pca_reduce(hog_tr, hog_te, 0.95)
print('PCA 95%:', hog_tr.shape[1], '->', hog_pca_tr.shape[1], 'chiều')
for feat_name, (Xtr, Xte) in {'HOG': (hog_tr, hog_te), 'HOG+PCA95': (hog_pca_tr, hog_pca_te)}.items():
    for clf_name in ['logreg', 'linsvm', 'svm_rbf', 'rf']:
        r, _ = IM.fit_eval(IM.make_classifier(clf_name), Xtr, y_train, Xte, y_test)
        log.add(features=feat_name, classifier=clf_name, dim=Xtr.shape[1], **r)
        print(f"{feat_name:<10} {clf_name:<8} acc={r['accuracy']:.4f} f1={r['f1']:.4f} ({r['fit_seconds']}s)")
df_hog = log.save()

## 4. Pipeline hiện đại: đặc trưng từ mô hình pretrained → lưu `.npy`

Dùng backbone **pretrained trên ImageNet** (bỏ lớp phân loại cuối) để lấy vector đặc trưng:

| Mô hình | Loại | Dim |
|---|---|---|
| ResNet-50 | CNN | 2048 |
| EfficientNet-B0 | CNN | 1280 |
| ViT-B/16 | Transformer | 768 |

Cấu hình được: kích thước resize (`img_size`) và tên mô hình. Ngoài ra thử **ResNet-50 ở 160×160** để xem ảnh hưởng của độ phân giải (ViT-B/16 cố định 224).

In [ ]:
configs = [('resnet50', 224), ('efficientnet_b0', 224), ('vit_b_16', 224), ('resnet50', 160)]
feats = {}
for name, size in configs:
    feats[(name, size)] = IM.extract_and_save(train_ds, test_ds, name, size)

In [ ]:
# Nạp lại đặc trưng từ file .npy và huấn luyện các bộ phân loại
log2 = ResultLogger('image_deep_features')
for name, size in configs:
    tag = f'{name}_{size}'
    Xtr, Xte = load_features(f'img_{tag}_train'), load_features(f'img_{tag}_test')
    for clf_name in ['logreg', 'linsvm', 'svm_rbf', 'rf']:
        r, _ = IM.fit_eval(IM.make_classifier(clf_name), Xtr, y_train, Xte, y_test)
        log2.add(backbone=name, img_size=size, classifier=clf_name, dim=Xtr.shape[1], **r)
        print(f"{tag:<22} {clf_name:<8} acc={r['accuracy']:.4f} f1={r['f1']:.4f} ({r['fit_seconds']}s)")
df_deepfeat = log2.save()

In [ ]:
import seaborn as sns
pv = df_deepfeat.assign(cfg=df_deepfeat.backbone + '_' + df_deepfeat.img_size.astype(str)).pivot(index='cfg', columns='classifier', values='accuracy')
plt.figure(figsize=(7, 3.5)); sns.heatmap(pv, annot=True, fmt='.3f', cmap='YlGnBu'); plt.title('Accuracy — backbone × bộ phân loại')
save_fig('img_deepfeat_heatmap')

## 5. (Điểm cộng) Pipeline học sâu đầu-cuối

**5a.** MLP PyTorch trên đặc trưng ViT/ResNet đã lưu.  
**5b.** **Fine-tune toàn bộ ResNet-50** (5 epoch, augmentation, label smoothing, OneCycle).

In [ ]:
log3 = ResultLogger('image_deep')
for name, size in [('resnet50', 224), ('vit_b_16', 224)]:
    tag = f'{name}_{size}'
    Xtr, Xte = load_features(f'img_{tag}_train'), load_features(f'img_{tag}_test')
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    r, _, _ = dl.train_mlp((Xtr - mu) / sd, y_train, (Xte - mu) / sd, y_test, hidden=(512,), dropout=0.4, verbose=False)
    log3.add(method=f'MLP trên {tag}', **r); print(f'MLP trên {tag}:', {k: round(v, 4) for k, v in r.items()})

In [ ]:
if torch.cuda.is_available():
    r_ft, pred_ft = IM.finetune_cnn(train_ds, test_ds, n_classes=len(class_names), arch='resnet50', epochs=5)
else:
    print('Không có GPU -> fine-tune ResNet-18 1 epoch để minh hoạ. Hãy bật GPU (T4) để có kết quả đầy đủ.')
    r_ft, pred_ft = IM.finetune_cnn(train_ds, test_ds, n_classes=len(class_names), arch='resnet18', epochs=1, img_size=128)
log3.add(method='Fine-tune ResNet-50 (end-to-end)', **r_ft)
print({k: round(v, 4) for k, v in r_ft.items()})
df_deep = log3.save(); df_deep

## 6. Tổng hợp & phân tích

In [ ]:
best_hog = df_hog.sort_values('accuracy', ascending=False).iloc[0]
best_df = df_deepfeat.sort_values('accuracy', ascending=False).iloc[0]
summary = pd.DataFrame([
    {'approach': 'Truyền thống', 'method': f"{best_hog['features']} + {best_hog['classifier']}", 'accuracy': best_hog['accuracy'], 'f1': best_hog['f1']},
    {'approach': 'Pretrained features + ML cổ điển', 'method': f"{best_df['backbone']}_{best_df['img_size']} + {best_df['classifier']}", 'accuracy': best_df['accuracy'], 'f1': best_df['f1']},
] + [{'approach': 'Học sâu', 'method': r['method'], 'accuracy': r['accuracy'], 'f1': r['f1']} for _, r in df_deep.iterrows()])
summary.to_csv('reports/results_image_summary.csv', index=False)
plt.figure(figsize=(9, 3.8)); plt.barh(summary['method'], summary['accuracy'], color=plt.cm.viridis(np.linspace(0.1, 0.9, len(summary))))
plt.xlim(0, 1); plt.xlabel('Accuracy'); plt.title('So sánh các cách tiếp cận (Oxford-IIIT Pet)')
for i, v in enumerate(summary['accuracy']): plt.text(v + 0.005, i, f'{v:.3f}', va='center')
save_fig('img_summary_bar'); summary

In [ ]:
# Phân tích lỗi: mô hình pretrained-features tốt nhất
tag = f"{best_df['backbone']}_{best_df['img_size']}"
Xtr, Xte = load_features(f'img_{tag}_train'), load_features(f'img_{tag}_test')
r, pred = IM.fit_eval(IM.make_classifier(best_df['classifier']), Xtr, y_train, Xte, y_test)
plot_confusion(y_test, pred, labels=class_names, title=f'Confusion matrix — {tag} + {best_df["classifier"]}', name='img_confusion_best')
cm = pd.crosstab(pd.Series(np.array(class_names)[y_test], name='thực tế'), pd.Series(np.array(class_names)[pred], name='dự đoán'))
cm = cm.reindex(index=class_names, columns=class_names, fill_value=0)
pairs = [(a, b, cm.loc[a, b]) for a in cm.index for b in cm.columns if a != b]
print('Cặp giống bị nhầm nhiều nhất:')
for a, b, n in sorted(pairs, key=lambda x: -x[2])[:8]: print(f'  {a} -> {b}: {n}')
wrong = np.where(pred != y_test)[0][:12]
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
from PIL import Image
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(Image.open(test_ds._images[i]).convert('RGB')); ax.axis('off')
    ax.set_title(f'T: {class_names[y_test[i]]}\nP: {class_names[pred[i]]}', fontsize=7)
save_fig('img_errors')

## 7. Kết luận *(chỉnh theo kết quả thực tế)*

- **HOG + ML cổ điển** cho kết quả thấp (đặc trưng cạnh/hình dạng cục bộ không đủ cho bài toán fine-grained 37 lớp); PCA giảm chiều nhưng không cải thiện nhiều.
- **Đặc trưng pretrained** cải thiện **rất mạnh** (chuyển giao tri thức từ ImageNet); mô hình tuyến tính (Logistic Regression/Linear SVM) trên đặc trưng chuẩn hoá thường tốt nhất, Random Forest kém trên vector chiều cao.
- **ViT-B/16 vs ResNet-50 vs EfficientNet-B0:** *…*; **độ phân giải 160 vs 224:** *…*
- **Fine-tune end-to-end** thường cao nhất nhưng tốn thời gian GPU hơn nhiều so với trích đặc trưng một lần.
- **Lỗi** tập trung ở các giống nhìn rất giống nhau.
- **Hạn chế:** không dùng test-time augmentation, chưa thử SIFT/bag-of-visual-words, chỉ một lần chia train/test cố định.

Đặc trưng đã lưu ở `features/img_*.npy`; bảng kết quả ở `reports/results_image_*.csv`.